# Análise Exploratória (EDA)

**Projeto:** Social Wave — Otimização de Campanhas de Marketing  
**Fase:** Diagnóstico (Fase 1)  
**Objetivo:** Entender o passado antes de propor o futuro  

---

## Perguntas que este notebook responde

| # | Pergunta | O que descobrimos | Seção |
|---|----------|-------------------|-------|
| **1.1** | Qual o **CPA médio por canal** e como ele evoluiu ao longo do tempo? | Canais eficientes vs. ineficientes | 3.1 |
| **1.2** | Qual o **share de gasto** e **share de conversões** por canal hoje? | Desbalanceamento entre investimento e retorno | 3.2 |
| **1.3** | Existe **correlação entre aumento de gasto e elevação de CPA** dentro de cada canal? | Sinais de retornos decrescentes | 3.3 |
| **1.4** | Qual o **CTR e taxa de conversão** por canal? | Onde o funil 'vaza' | 3.4 |

---

## Dados de Entrada
Base limpa do notebook `notebook_00_contexto_dicionario_e_tratamento.ipynb`  
Período: Outubro a Dezembro de 2023  
Registros: [será preenchido após carga]  
Canais: 6 (Meta, Google, YouTube, Twitter, TikTok, LinkedIn)

---

## Entregáveis
- Tabela resumo por canal (CPA, CTR, Taxa de Conversão, Share)
- Gráficos: evolução temporal, distribuição, correlações
- Insights iniciais para o modelo de otimização

## Setup e Carregamento da Base processada no Notebook 00

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# --- Configuracoes visuais ---
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# --- Configuracoes de exibicao ---
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# --- Seed para reprodutibilidade ---
np.random.seed(42)

# ============================================================
# CARREGAR BASE PROCESSADA (PICKLE)
# ============================================================

PICKLE_PATH = r'C:\Users\ricar\Documents\projetos\social-wave-campaign-optimization\data\campanhas_base_processada.pkl'

try:
    df = pd.read_pickle(PICKLE_PATH)
    
    print('=' * 60)
    print('📊 BASE PROCESSADA CARREGADA')
    print('=' * 60)
    print(f'Fonte: {PICKLE_PATH}')
    print(f'Dimensoes: {df.shape[0]:,} registros x {df.shape[1]} colunas')
    print(f'Periodo: {df['Data da Coleta'].min()} a {df['Data da Coleta'].max()}')
    
    print(f'\n✅ Tipos preservados:')
    print(f'   • Data da Coleta: {df['Data da Coleta'].dtype}')
    print(f'   • Canal: {df['Canal'].dtype}')
    print(f'   • Gasto: {df['Gasto'].dtype}')
    print(f'   • Conversao: {df['Conversao'].dtype}')
    
    print(f'\n📋 Colunas disponiveis:')
    print(f'   {', '.join(df.columns.tolist())}')
    
except FileNotFoundError:
    print(f'❌ Base nao encontrada: {PICKLE_PATH}')
    print('   Execute o Notebook 00 primeiro para gerar a base limpa.')
    raise

print(f'\n📊 Primeiras linhas:')
display(df.head())

📊 BASE PROCESSADA CARREGADA
Fonte: C:\Users\ricar\Documents\projetos\social-wave-campaign-optimization\data\campanhas_base_processada.pkl
Dimensoes: 23,315 registros x 9 colunas
Periodo: 2023-10-01 11:00:00 a 2023-12-24 23:00:00

✅ Tipos preservados:
   • Data da Coleta: datetime64[us]
   • Canal: str
   • Gasto: float64
   • Conversao: int64

📋 Colunas disponiveis:
   ID, Canal, Data da Coleta, Impressoes Disponiveis, Consultas Correspondentes, Impressoes, Cliques, Gasto, Conversao

📊 Primeiras linhas:


,ID,Canal,Data da Coleta,Impressoes Disponiveis,Consultas Correspondentes,Impressoes,Cliques,Gasto,Conversao
0,ADSXJ0000001,Canal1,2023-11-02 03:00:00,82132,61776,43673,5,37.76,2
1,ADSXJ0000002,Canal1,2023-11-02 03:00:00,41889,33512,23919,4,18.76,2
2,ADSXJ0000003,Canal2,2023-12-18 04:00:00,2438114,1611096,1215019,1314,2481.57,44
3,ADSXJ0000004,Canal2,2023-12-19 04:00:00,2429576,1608711,1212549,1343,2513.02,36
4,ADSXJ0000005,Canal2,2023-12-18 03:00:00,2863336,1929554,1456197,1566,2912.62,30


## Cálculo de Métricas Derivadas

Objetivo: Transformar dados brutos em KPIs de marketing.

Fórmulas: CTR, Taxa de Conversão, CPC, CPA, CPM